In [89]:
import torch
print("CUDA beschikbaar:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA beschikbaar: True
Device: Tesla T4


In [90]:
import random

def set_seed(seed=42):
    """Zorg voor reproduceerbare resultaten."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Voor strikt deterministisch gedrag op CUDA:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [91]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

In [92]:
URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv"
df = pd.read_csv(URL)
df = df.dropna(subset=["smiles"]).reset_index(drop=True)

print(f"Aantal moleculen: {len(df)}")
print(f"Class balance:\n{df['p_np'].value_counts()}")

Aantal moleculen: 2050
Class balance:
p_np
1    1567
0     483
Name: count, dtype: int64


In [93]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict
from sklearn.model_selection import GroupKFold

def get_scaffold(smi):
    """Bereken Bemis-Murcko scaffold van een SMILES-string."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return ""
    scaffold_mol = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold_mol)

smiles = df["smiles"].tolist()
labels = df["p_np"].astype(int).tolist()

# --- Bereken scaffolds voor alle moleculen ---
print("Berekenen scaffolds...")
scaffolds = [get_scaffold(s) for s in smiles]
print(f"Unieke scaffolds: {len(set(scaffolds))} (uit {len(smiles)} moleculen)")

# Groepeer molecule-indices per scaffold
scaffold_to_indices = defaultdict(list)
for i, sc in enumerate(scaffolds):
    scaffold_to_indices[sc].append(i)

# Sorteer scaffold-groepen van groot naar klein
scaffold_groups = sorted(scaffold_to_indices.values(), key=lambda g: -len(g))

# --- Verdeel scaffolds over train+val en test, met class-balance-bewaking ---
n_total = len(smiles)
test_size_target = int(0.10 * n_total)
overall_pos_rate = np.mean(labels)   # ~0.76 voor BBBP

trainval_idx, test_idx = [], []
trainval_labels_so_far = []
test_labels_so_far     = []

for group in scaffold_groups:
    group_labels = [labels[i] for i in group]

    if len(test_idx) < test_size_target:
        # Kijk of test deze groep aankan zonder de balans te veel te verstoren
        candidate_test = test_labels_so_far + group_labels
        test_pos_rate = np.mean(candidate_test)

        if (abs(test_pos_rate - overall_pos_rate) < 0.10
                or len(test_idx) < 0.5 * test_size_target):
            test_idx.extend(group)
            test_labels_so_far.extend(group_labels)
            continue

    trainval_idx.extend(group)
    trainval_labels_so_far.extend(group_labels)

# --- Maak de bijbehorende lijsten ---
trainval_smiles    = [smiles[i]    for i in trainval_idx]
trainval_labels    = [labels[i]    for i in trainval_idx]
trainval_scaffolds = [scaffolds[i] for i in trainval_idx]  # nodig voor CV
test_smiles        = [smiles[i]    for i in test_idx]
test_labels        = [labels[i]    for i in test_idx]

# --- Rapporteer ---
print(f"\nTrain+Val: {len(trainval_smiles)} | Test: {len(test_smiles)}")
print(f"Class balance train+val: {np.bincount(trainval_labels)} "
      f"(pos rate: {np.mean(trainval_labels):.3f})")
print(f"Class balance test:      {np.bincount(test_labels)} "
      f"(pos rate: {np.mean(test_labels):.3f})")
print(f"Overall pos rate:        {overall_pos_rate:.3f}")

# --- Sanity check op scaffold-disjunctie ---
trainval_scaffold_set = set(trainval_scaffolds)
test_scaffold_set     = {scaffolds[i] for i in test_idx}
overlap = trainval_scaffold_set & test_scaffold_set
print(f"\nScaffold overlap train+val ↔ test: {len(overlap)} "
      f"(zou 0 moeten zijn)")

Berekenen scaffolds...


[14:56:53] Explicit valence for atom # 1 N, 4, is greater than permitted
[14:56:53] WARNING: not removing hydrogen atom without neighbors
[14:56:53] Explicit valence for atom # 6 N, 4, is greater than permitted
[14:56:53] WARNING: not removing hydrogen atom without neighbors
[14:56:53] WARNING: not removing hydrogen atom without neighbors
[14:56:53] WARNING: not removing hydrogen atom without neighbors
[14:56:53] WARNING: not removing hydrogen atom without neighbors
[14:56:53] WARNING: not removing hydrogen atom without neighbors
[14:56:53] WARNING: not removing hydrogen atom without neighbors
[14:56:53] Explicit valence for atom # 6 N, 4, is greater than permitted
[14:56:53] WARNING: not removing hydrogen atom without neighbors
[14:56:53] WARNING: not removing hydrogen atom without neighbors
[14:56:53] WARNING: not removing hydrogen atom without neighbors
[14:56:53] WARNING: not removing hydrogen atom without neighbors
[14:56:53] Explicit valence for atom # 11 N, 4, is greater than pe

Unieke scaffolds: 1102 (uit 2050 moleculen)

Train+Val: 1803 | Test: 247
Class balance train+val: [ 435 1368] (pos rate: 0.759)
Class balance test:      [ 48 199] (pos rate: 0.806)
Overall pos rate:        0.764

Scaffold overlap train+val ↔ test: 0 (zou 0 moeten zijn)


[14:56:53] WARNING: not removing hydrogen atom without neighbors
[14:56:53] WARNING: not removing hydrogen atom without neighbors
[14:56:53] WARNING: not removing hydrogen atom without neighbors
[14:56:54] WARNING: not removing hydrogen atom without neighbors
[14:56:54] WARNING: not removing hydrogen atom without neighbors
[14:56:54] WARNING: not removing hydrogen atom without neighbors
[14:56:54] WARNING: not removing hydrogen atom without neighbors
[14:56:54] WARNING: not removing hydrogen atom without neighbors
[14:56:54] WARNING: not removing hydrogen atom without neighbors
[14:56:54] WARNING: not removing hydrogen atom without neighbors


In [94]:
all_chars = set()
for smi in trainval_smiles:   # was: train_smiles
    all_chars.update(smi)

sorted_chars = sorted(all_chars)
char_to_idx = {"<PAD>": 0, "<UNK>": 1}
for i, c in enumerate(sorted_chars, start=2):
    char_to_idx[c] = i

idx_to_char = {i: c for c, i in char_to_idx.items()}
vocab_size = len(char_to_idx)
print(f"Vocab size: {vocab_size}")

Vocab size: 41


In [95]:
lengths = [len(s) for s in trainval_smiles]   # was: train_smiles
print(f"SMILES lengtes — min: {min(lengths)}, max: {max(lengths)}, "
      f"mean: {np.mean(lengths):.1f}, 95-percentiel: {int(np.percentile(lengths, 95))}")

SMILES lengtes — min: 5, max: 400, mean: 55.1, 95-percentiel: 108


In [96]:
MAX_LENGTH = 200

def encode_smiles(smi, char_to_idx, max_length=MAX_LENGTH):
    """Zet een SMILES-string om in een lijst integers van vaste lengte."""
    smi = smi[:max_length]
    ids = [char_to_idx.get(c, char_to_idx["<UNK>"]) for c in smi]
    ids = ids + [char_to_idx["<PAD>"]] * (max_length - len(ids))
    return ids


class SMILES_CNN_Dataset(Dataset):
    def __init__(self, smiles, labels, char_to_idx, max_length=MAX_LENGTH):
        self.smiles = smiles
        self.labels = labels
        self.char_to_idx = char_to_idx
        self.max_length = max_length

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        ids = encode_smiles(self.smiles[idx], self.char_to_idx, self.max_length)
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

print(f"Dataset-klasse gedefinieerd. MAX_LENGTH = {MAX_LENGTH}")

Dataset-klasse gedefinieerd. MAX_LENGTH = 200


In [97]:
class SMILES_CNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_filters=64,
                 kernel_sizes=(3, 5, 7), dropout=0.5, num_classes=2,
                 class_weights=None):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k, padding=k // 2)
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

        if class_weights is not None:
            self.register_buffer(
                "class_weights",
                torch.tensor(class_weights, dtype=torch.float32)
            )
        else:
            self.class_weights = None

    def forward(self, input_ids, labels=None):
        x = self.embedding(input_ids)
        x = x.permute(0, 2, 1)

        conv_outputs = []
        for conv in self.convs:
            c = F.relu(conv(x))
            p = F.max_pool1d(c, c.size(2)).squeeze(2)
            conv_outputs.append(p)

        x = torch.cat(conv_outputs, dim=1)
        x = self.dropout(x)
        logits = self.fc(x)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels, weight=self.class_weights)

        return {"loss": loss, "logits": logits}


# Deze regels staan BUITEN de class — geen inspringing
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [98]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

def evaluate(model, loader, device):
    """Evalueer model op een loader, geeft loss/accuracy/AUC terug."""
    model.eval()
    total_loss = 0.0
    all_logits, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            out = model(input_ids, labels=labels)
            total_loss += out["loss"].item() * input_ids.size(0)
            all_logits.append(out["logits"].cpu())
            all_labels.append(labels.cpu())

    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels).numpy()
    preds = logits.argmax(dim=1).numpy()
    probs = torch.softmax(logits, dim=1)[:, 1].numpy()

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(labels, preds),
        "roc_auc":  roc_auc_score(labels, probs),
    }


def train_model(model, train_loader, val_loader, epochs=30, lr=1e-3,
                weight_decay=1e-4, verbose=True):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_auc = 0.0
    best_state = None
    history = []

    if verbose:
        print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>9} | {'Val Acc':>7} | {'Val AUC':>7}")
        print("-" * 55)

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            out = model(input_ids, labels=labels)
            out["loss"].backward()
            optimizer.step()
            epoch_loss += out["loss"].item() * input_ids.size(0)

        train_loss = epoch_loss / len(train_loader.dataset)
        val_metrics = evaluate(model, val_loader, device)
        history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})

        if verbose:
            print(f"{epoch:>5} | {train_loss:>10.4f} | {val_metrics['loss']:>9.4f} "
                  f"| {val_metrics['accuracy']:>7.4f} | {val_metrics['roc_auc']:>7.4f}")

        if val_metrics["roc_auc"] > best_val_auc:
            best_val_auc = val_metrics["roc_auc"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    if verbose:
        print(f"\nBeste val AUC: {best_val_auc:.4f}")
    return history

print("Functies 'evaluate' en 'train_model' gedefinieerd.")

Functies 'evaluate' en 'train_model' gedefinieerd.


In [99]:
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_class_weight
from itertools import product
import time

N_FOLDS = 5
EPOCHS = 30
BATCH_SIZE = 32

# ----------------------------
# Hyperparameter grid
# ----------------------------
param_grid = {
    "lr":          [1e-4, 1e-3, 1e-2],
    "dropout":     [0.3, 0.5],
    "num_filters": [32, 64],
}

keys = list(param_grid.keys())
combos = [dict(zip(keys, values)) for values in product(*param_grid.values())]
print(f"Aantal combinaties: {len(combos)}")
print(f"Verwachte trainruns: {len(combos) * N_FOLDS}\n")

trainval_smiles_arr = np.array(trainval_smiles)
trainval_labels_arr = np.array(trainval_labels)

# ----------------------------
# CV-runner voor één combinatie
# ----------------------------
def run_cv_for_combo(hyperparams):
    """Run 5-fold scaffold CV met gegeven hyperparams (silent)."""
    gkf = GroupKFold(n_splits=N_FOLDS)
    val_aucs, models = [], []
    val_probs_all, val_labels_all = [], []

    for fold, (train_idx, val_idx) in enumerate(
        gkf.split(trainval_smiles_arr, trainval_labels_arr, groups=trainval_scaffolds),
        start=1
    ):
        fold_train_smiles = trainval_smiles_arr[train_idx].tolist()
        fold_train_labels = trainval_labels_arr[train_idx].tolist()
        fold_val_smiles   = trainval_smiles_arr[val_idx].tolist()
        fold_val_labels   = trainval_labels_arr[val_idx].tolist()

        cw = compute_class_weight(
            "balanced", classes=np.array([0, 1]),
            y=np.array(fold_train_labels)
        )

        train_ds = SMILES_CNN_Dataset(fold_train_smiles, fold_train_labels, char_to_idx)
        val_ds   = SMILES_CNN_Dataset(fold_val_smiles,   fold_val_labels,   char_to_idx)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

        model = SMILES_CNN(
            vocab_size=vocab_size,
            num_filters=hyperparams["num_filters"],
            dropout=hyperparams["dropout"],
            class_weights=cw
        ).to(device)

        history = train_model(
            model, train_loader, val_loader,
            epochs=EPOCHS, lr=hyperparams["lr"], verbose=False
        )

        best_val = max(history, key=lambda h: h["roc_auc"])
        val_aucs.append(best_val["roc_auc"])
        models.append(model)

        # OOF predictions verzamelen
        model.eval()
        probs, lbls = [], []
        with torch.no_grad():
            for batch in val_loader:
                out = model(batch["input_ids"].to(device))
                probs.append(torch.softmax(out["logits"], dim=1)[:, 1].cpu().numpy())
                lbls.append(batch["labels"].numpy())
        val_probs_all.append(np.concatenate(probs))
        val_labels_all.append(np.concatenate(lbls))

    return np.mean(val_aucs), np.std(val_aucs), models, val_probs_all, val_labels_all

# ----------------------------
# Grid search loop
# ----------------------------
set_seed(42)
results = []
start_time = time.time()

for i, combo in enumerate(combos, start=1):
    print(f"[{i:2d}/{len(combos)}] {combo}", end="  → ")
    t0 = time.time()
    mean_auc, std_auc, models, val_probs, val_labels = run_cv_for_combo(combo)
    elapsed = time.time() - t0
    print(f"Val AUC = {mean_auc:.4f} ± {std_auc:.4f}  ({elapsed:.1f}s)")
    results.append({
        "hyperparams": combo,
        "mean_auc":    mean_auc,
        "std_auc":     std_auc,
        "models":      models,
        "val_probs":   val_probs,
        "val_labels":  val_labels,
    })

total = time.time() - start_time
print(f"\nTotale tijd grid search: {total/60:.1f} min")

# ----------------------------
# Top 5 overzicht
# ----------------------------
results_sorted = sorted(results, key=lambda r: -r["mean_auc"])

print("\n========== TOP 5 HYPERPARAMETER COMBINATIES ==========")
print(f"{'Rank':>4} | {'Mean AUC':>10} | {'Std':>7} | Hyperparams")
print("-" * 75)
for rank, r in enumerate(results_sorted[:5], start=1):
    print(f"{rank:>4} | {r['mean_auc']:>10.4f} | {r['std_auc']:>7.4f} | {r['hyperparams']}")

# ----------------------------
# Beste combinatie selecteren voor Cell 11
# ----------------------------
best = results_sorted[0]
print(f"\n>>> Beste hyperparameters: {best['hyperparams']}")
print(f">>> Mean val ROC-AUC: {best['mean_auc']:.4f} ± {best['std_auc']:.4f}")

# Variabelen die Cell 11 verwacht
fold_models    = best["models"]
all_val_probs  = best["val_probs"]
all_val_labels = best["val_labels"]

Aantal combinaties: 12
Verwachte trainruns: 60

[ 1/12] {'lr': 0.0001, 'dropout': 0.3, 'num_filters': 32}  → Val AUC = 0.9413 ± 0.0191  (38.7s)
[ 2/12] {'lr': 0.0001, 'dropout': 0.3, 'num_filters': 64}  → Val AUC = 0.9464 ± 0.0165  (41.0s)
[ 3/12] {'lr': 0.0001, 'dropout': 0.5, 'num_filters': 32}  → Val AUC = 0.9417 ± 0.0183  (38.3s)
[ 4/12] {'lr': 0.0001, 'dropout': 0.5, 'num_filters': 64}  → Val AUC = 0.9459 ± 0.0166  (40.1s)
[ 5/12] {'lr': 0.001, 'dropout': 0.3, 'num_filters': 32}  → Val AUC = 0.9519 ± 0.0141  (38.1s)
[ 6/12] {'lr': 0.001, 'dropout': 0.3, 'num_filters': 64}  → Val AUC = 0.9525 ± 0.0136  (42.1s)
[ 7/12] {'lr': 0.001, 'dropout': 0.5, 'num_filters': 32}  → Val AUC = 0.9517 ± 0.0192  (38.0s)
[ 8/12] {'lr': 0.001, 'dropout': 0.5, 'num_filters': 64}  → Val AUC = 0.9511 ± 0.0162  (48.4s)
[ 9/12] {'lr': 0.01, 'dropout': 0.3, 'num_filters': 32}  → Val AUC = 0.9488 ± 0.0137  (38.1s)
[10/12] {'lr': 0.01, 'dropout': 0.3, 'num_filters': 64}  → Val AUC = 0.9499 ± 0.0115  (39.8s)


In [100]:
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    classification_report, f1_score, average_precision_score
)

# === Stap 1: vind optimale threshold op out-of-fold val data ===
# Belangrijk: NIET op test set zoeken — anders is het cherry-picking
oof_probs  = np.concatenate(all_val_probs)
oof_labels = np.concatenate(all_val_labels)

thresholds = np.linspace(0.05, 0.95, 91)
val_f1s = [f1_score(oof_labels, (oof_probs > t).astype(int), average="macro")
           for t in thresholds]
best_t = thresholds[int(np.argmax(val_f1s))]
print(f"Optimale threshold (op out-of-fold val): {best_t:.3f}")
print(f"Macro F1 op val bij deze threshold:      {max(val_f1s):.4f}")

# === Stap 2: ensemble-voorspelling op test set ===
test_ds = SMILES_CNN_Dataset(test_smiles, test_labels, char_to_idx)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

all_probs_per_fold = []
for m in fold_models:
    m.eval()
    probs = []
    with torch.no_grad():
        for batch in test_loader:
            out = m(batch["input_ids"].to(device))
            probs.append(torch.softmax(out["logits"], dim=1)[:, 1].cpu().numpy())
    all_probs_per_fold.append(np.concatenate(probs))

mean_probs = np.mean(all_probs_per_fold, axis=0)
y_true = np.array(test_labels)

# === Stap 3: rapporteer met DEFAULT threshold (0.5) ===
y_pred_default = (mean_probs > 0.5).astype(int)
print("\n=== Test resultaten — default threshold (0.5) ===")
print(f"Accuracy: {accuracy_score(y_true, y_pred_default):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_true, mean_probs):.4f}")
print(f"PR-AUC:   {average_precision_score(y_true, mean_probs):.4f}")
print(f"\nConfusion matrix:\n{confusion_matrix(y_true, y_pred_default)}")
print(f"\n{classification_report(y_true, y_pred_default, target_names=['geen BBB', 'wel BBB'])}")

# === Stap 4: rapporteer met TUNED threshold ===
y_pred_tuned = (mean_probs > best_t).astype(int)
print(f"\n=== Test resultaten — tuned threshold ({best_t:.3f}) ===")
print(f"Accuracy: {accuracy_score(y_true, y_pred_tuned):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_true, mean_probs):.4f}  (threshold-onafhankelijk)")
print(f"PR-AUC:   {average_precision_score(y_true, mean_probs):.4f}  (threshold-onafhankelijk)")
print(f"\nConfusion matrix:\n{confusion_matrix(y_true, y_pred_tuned)}")
print(f"\n{classification_report(y_true, y_pred_tuned, target_names=['geen BBB', 'wel BBB'])}")

Optimale threshold (op out-of-fold val): 0.340
Macro F1 op val bij deze threshold:      0.8507

=== Test resultaten — default threshold (0.5) ===
Accuracy: 0.8097
ROC-AUC:  0.9447
PR-AUC:   0.9869

Confusion matrix:
[[ 48   0]
 [ 47 152]]

              precision    recall  f1-score   support

    geen BBB       0.51      1.00      0.67        48
     wel BBB       1.00      0.76      0.87       199

    accuracy                           0.81       247
   macro avg       0.75      0.88      0.77       247
weighted avg       0.90      0.81      0.83       247


=== Test resultaten — tuned threshold (0.340) ===
Accuracy: 0.8623
ROC-AUC:  0.9447  (threshold-onafhankelijk)
PR-AUC:   0.9869  (threshold-onafhankelijk)

Confusion matrix:
[[ 40   8]
 [ 26 173]]

              precision    recall  f1-score   support

    geen BBB       0.61      0.83      0.70        48
     wel BBB       0.96      0.87      0.91       199

    accuracy                           0.86       247
   macro avg    